# 02 Data Cleaning

This notebook cleans and prepares the electronics store ecommerce dataset for later analysis.

## 1. Import libraries and define file paths

We start by importing pandas and setting the input and output file paths for this project.

In [1]:
from pathlib import Path
import pandas as pd

INPUT_PATH = Path("..") / "data" / "events.csv"
OUTPUT_PATH = Path("..") / "outputs" / "events_cleaned.csv"

print(f"Input file: {INPUT_PATH.resolve()}")
print(f"Output file: {OUTPUT_PATH.resolve()}")

Input file: C:\Users\jayso\.vscode\internships projects\product_analytics_cohort\data\events.csv
Output file: C:\Users\jayso\.vscode\internships projects\product_analytics_cohort\outputs\events_cleaned.csv


## 2. Load the raw dataset

We read the raw CSV file from the `data` folder.

In [2]:
df = pd.read_csv(INPUT_PATH)
original_shape = df.shape
print("Original shape:", original_shape)
df.head()

Original shape: (885129, 9)


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2020-09-24 11:57:06 UTC,view,1996170,2144415922528452715,electronics.telephone,NaN,31.90,1515915625519388267,LJuJVLEjPT
1,2020-09-24 11:57:26 UTC,view,139905,2144415926932472027,computers.components.cooler,zalman,17.16,1515915625519380411,tdicluNnRY
2,2020-09-24 11:57:27 UTC,view,215454,2144415927158964449,NaN,NaN,9.81,1515915625513238515,4TMArHtXQy
3,2020-09-24 11:57:33 UTC,view,635807,2144415923107266682,computers.peripherals.printer,pantum,113.81,1515915625519014356,aGFYrNgC08
4,2020-09-24 11:57:36 UTC,view,3658723,2144415921169498184,NaN,cameronsino,15.87,1515915625510743344,aa4mmk0kwQ


## 3. Convert `event_time` to datetime

This makes the time column easier to work with and allows us to create date-based fields later.

In [3]:
df["event_time"] = pd.to_datetime(df["event_time"], errors="coerce")
print("Missing event_time values after conversion:", df["event_time"].isna().sum())

Missing event_time values after conversion: 0


## 4. Check for duplicate rows

We count duplicate rows first, then remove them so they do not inflate later metrics.

In [4]:
duplicate_rows = df.duplicated().sum()
print("Duplicate rows found:", duplicate_rows)

df = df.drop_duplicates().copy()
print("Shape after removing duplicates:", df.shape)

Duplicate rows found: 655


Shape after removing duplicates: (884474, 9)


## 5. Check missing or invalid values in key columns

Here we inspect the most important fields we will rely on later for analytics.

In [5]:
key_columns = [
    "event_time",
    "event_type",
    "product_id",
    "category_code",
    "brand",
    "price",
    "user_id",
    "user_session",
]

missing_key_values = df[key_columns].isna().sum().sort_values(ascending=False)
print("Missing values in key columns:")
print(missing_key_values)

print("\nUnique event types:")
print(sorted(df["event_type"].dropna().unique().tolist()))

invalid_user_ids = (df["user_id"] <= 0).sum()
invalid_product_ids = (df["product_id"] <= 0).sum()

print("\nInvalid user_id count (<= 0):", invalid_user_ids)
print("Invalid product_id count (<= 0):", invalid_product_ids)

Missing values in key columns:
category_code    236047
brand            212232
user_session        162
event_time            0
product_id            0
event_type            0
price                 0
user_id               0
dtype: int64

Unique event types:


['cart', 'purchase', 'view']

Invalid user_id count (<= 0): 0
Invalid product_id count (<= 0): 0


## 6. Make sure `price` is numeric

If needed, we convert the column to numeric and check for missing or non-positive values.

In [6]:
df["price"] = pd.to_numeric(df["price"], errors="coerce")

missing_price_count = df["price"].isna().sum()
non_positive_price_count = (df["price"] <= 0).sum()

print("Missing price values after conversion:", missing_price_count)
print("Non-positive price values:", non_positive_price_count)

Missing price values after conversion: 0
Non-positive price values: 0


## 7. Clean the most important fields

For analytics, we keep rows with the essential identifiers and valid time values. We also handle missing category and brand values in a practical way by filling them with `Unknown`, which keeps those records available for summaries and dashboards.

In [7]:
rows_before_cleaning = len(df)

df = df.dropna(subset=["event_time", "event_type", "product_id", "user_id", "price"]).copy()
df = df[df["price"] > 0].copy()

df["brand"] = df["brand"].fillna("Unknown")
df["category_code"] = df["category_code"].fillna("Unknown")
df["user_session"] = df["user_session"].fillna("missing_session")

rows_removed_during_cleaning = rows_before_cleaning - len(df)
print("Rows removed during cleaning:", rows_removed_during_cleaning)
print("Shape after cleaning:", df.shape)

Rows removed during cleaning: 0
Shape after cleaning: (884474, 9)


## 8. Create derived date columns

These fields will make it easier to summarize the data by day, month, and week later.

In [8]:
df["event_date"] = df["event_time"].dt.date
df["event_month"] = df["event_time"].dt.to_period("M").astype(str)
df["event_week"] = df["event_time"].dt.to_period("W").astype(str)

df[["event_time", "event_date", "event_month", "event_week"]].head()

C:\Users\jayso\AppData\Local\Temp\ipykernel_16216\3453866398.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["event_month"] = df["event_time"].dt.to_period("M").astype(str)


C:\Users\jayso\AppData\Local\Temp\ipykernel_16216\3453866398.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["event_week"] = df["event_time"].dt.to_period("W").astype(str)


,event_time,event_date,event_month,event_week
0,2020-09-24 11:57:06+00:00,2020-09-24,2020-09,2020-09-21/2020-09-27
1,2020-09-24 11:57:26+00:00,2020-09-24,2020-09,2020-09-21/2020-09-27
2,2020-09-24 11:57:27+00:00,2020-09-24,2020-09,2020-09-21/2020-09-27
3,2020-09-24 11:57:33+00:00,2020-09-24,2020-09,2020-09-21/2020-09-27
4,2020-09-24 11:57:36+00:00,2020-09-24,2020-09,2020-09-21/2020-09-27


## 9. Save the cleaned dataset

We save the cleaned file to the `outputs` folder so it can be reused in later notebooks.

In [9]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned dataset saved to: {OUTPUT_PATH.resolve()}")

Cleaned dataset saved to: C:\Users\jayso\.vscode\internships projects\product_analytics_cohort\outputs\events_cleaned.csv


## 10. Final cleaning summary

This short summary shows the final dataset size, how many duplicate rows were removed, what missing values remain in key columns, and where the cleaned file was saved.

In [10]:
remaining_missing = df[key_columns].isna().sum()

print("Final shape:", df.shape)
print("Duplicate rows removed:", duplicate_rows)
print("Missing values remaining in key columns:")
print(remaining_missing)
print("Output file path:", OUTPUT_PATH.resolve())

Final shape: (884474, 12)
Duplicate rows removed: 655
Missing values remaining in key columns:
event_time       0
event_type       0
product_id       0
category_code    0
brand            0
price            0
user_id          0
user_session     0
dtype: int64
Output file path: C:\Users\jayso\.vscode\internships projects\product_analytics_cohort\outputs\events_cleaned.csv
